# AI-Assisted Quality Inspection — Casting Defect Classification

**Student:** SARTAJ S  
**Dataset:** Real-Life Industrial Dataset of Casting Product (Kaggle)  
**Purpose:** Check the ZIP file, extract the dataset, inspect its folders, train a simple image-classification model, evaluate it, and save the model and training outputs.

## How to use
1. Upload this `.ipynb` notebook to Google Colab.
2. Run the cells from top to bottom.
3. When asked, upload the casting dataset ZIP file.
4. The notebook checks the ZIP before extracting it. If the archive is corrupt, it stops with a clear message.
5. At the end, download the output ZIP containing the trained model and results.

This notebook uses TensorFlow/Keras and does **not** overwrite your separate steel-surface notebook or model files.


## 1. Check the runtime

In [1]:
import os
import sys
import platform

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Working directory: /content


## 2. Upload the dataset ZIP

Run this cell and select the freshly downloaded Kaggle ZIP file. You do not need to rename it. The notebook will use the uploaded filename automatically.

In [3]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

zip_candidates = [
    Path("/content") / name
    for name in uploaded.keys()
    if name.lower().endswith(".zip")
]

if not zip_candidates:
    raise FileNotFoundError(
        "No ZIP file was uploaded. Run this cell again and choose the casting dataset ZIP."
    )

ZIP_PATH = str(zip_candidates[0])
print("Selected ZIP:", ZIP_PATH)
print("File size (MB):", round(os.path.getsize(ZIP_PATH) / (1024 * 1024), 2))

Saving archive (3).zip to archive (3) (1).zip
Selected ZIP: /content/archive (3) (1).zip
File size (MB): 100.1


## 3. Validate the ZIP before extraction

In [4]:
import zipfile

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(f"ZIP file not found: {ZIP_PATH}")

try:
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        file_list = archive.namelist()
        print("ZIP opened successfully.")
        print("Total entries:", len(file_list))
        bad_entry = archive.testzip()

        if bad_entry is not None:
            raise RuntimeError(
                f"The ZIP integrity check failed at: {bad_entry}\n"
                "Please download the dataset ZIP again from Kaggle and upload the new copy."
            )

        print("ZIP integrity check passed.")
except zipfile.BadZipFile as error:
    raise RuntimeError(
        "The uploaded file is not a valid ZIP archive. Download it again from Kaggle, "
        "then rerun this cell."
    ) from error

ZIP opened successfully.
Total entries: 8648
ZIP integrity check passed.


## 4. Extract the dataset

Extraction happens only after the ZIP integrity check passes.

In [5]:
import shutil
from pathlib import Path

EXTRACT_DIR = Path("/content/casting_dataset")

if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as archive:
    archive.extractall(EXTRACT_DIR)

print("Extracted to:", EXTRACT_DIR)
print("Top-level items:")
for item in list(EXTRACT_DIR.iterdir())[:20]:
    print(" -", item.name)

Extracted to: /content/casting_dataset
Top-level items:
 - casting_512x512
 - casting_data


## 5. Find the image folders and inspect the classes

The common casting dataset layout has folders such as `def_front` and `ok_front`. This cell detects class folders from the extracted data instead of assuming a fixed ZIP folder path.

In [6]:
from collections import Counter
from pathlib import Path

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
all_image_files = [
    p for p in EXTRACT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
]

if not all_image_files:
    raise RuntimeError(
        "No image files were found after extraction. Check that you uploaded the correct casting dataset ZIP."
    )

# Find leaf folders containing images. This handles nested ZIP layouts.
image_counts_by_folder = Counter(str(p.parent) for p in all_image_files)
class_folder_paths = [
    Path(folder) for folder, count in image_counts_by_folder.items() if count > 0
]

print("Total image files found:", len(all_image_files))
print("Folders containing images:", len(class_folder_paths))
print("\nSample folders and image counts:")
for folder in sorted(class_folder_paths)[:30]:
    count = sum(1 for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    print(f"{folder}: {count}")

Total image files found: 8648
Folders containing images: 6

Sample folders and image counts:
/content/casting_dataset/casting_512x512/casting_512x512/def_front: 781
/content/casting_dataset/casting_512x512/casting_512x512/ok_front: 519
/content/casting_dataset/casting_data/casting_data/test/def_front: 453
/content/casting_dataset/casting_data/casting_data/test/ok_front: 262
/content/casting_dataset/casting_data/casting_data/train/def_front: 3758
/content/casting_dataset/casting_data/casting_data/train/ok_front: 2875


## 6. Prepare the dataset

This cell identifies the `def_front` and `ok_front` class folders. If the dataset structure is different, it prints the detected folder names and stops instead of silently training on the wrong labels.

In [7]:
from pathlib import Path

# Locate folders named exactly as the expected binary classes.
defect_dirs = [p for p in EXTRACT_DIR.rglob("*") if p.is_dir() and p.name.lower() == "def_front"]
ok_dirs = [p for p in EXTRACT_DIR.rglob("*") if p.is_dir() and p.name.lower() == "ok_front"]

if not defect_dirs or not ok_dirs:
    folder_names = sorted({p.name for p in EXTRACT_DIR.rglob("*") if p.is_dir()})
    print("Expected class folders 'def_front' and 'ok_front' were not both found.")
    print("Detected folder names (sample):", folder_names[:60])
    raise RuntimeError(
        "Please check the extracted folder layout. The notebook stopped to avoid using incorrect labels."
    )

# Use the matching pair with the largest combined number of direct image files.
def direct_image_count(folder):
    return sum(
        1 for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

pairs = [(d, o) for d in defect_dirs for o in ok_dirs if d.parent == o.parent]
if pairs:
    DEFECT_DIR, OK_DIR = max(
        pairs,
        key=lambda pair: direct_image_count(pair[0]) + direct_image_count(pair[1])
    )
else:
    # Fall back to the most populated matching folder of each class.
    DEFECT_DIR = max(defect_dirs, key=direct_image_count)
    OK_DIR = max(ok_dirs, key=direct_image_count)

print("Defective class folder:", DEFECT_DIR)
print("OK class folder:", OK_DIR)
print("Defective images:", direct_image_count(DEFECT_DIR))
print("OK images:", direct_image_count(OK_DIR))

if direct_image_count(DEFECT_DIR) < 2 or direct_image_count(OK_DIR) < 2:
    raise RuntimeError("Not enough images in one or both class folders to train and validate a model.")

Defective class folder: /content/casting_dataset/casting_data/casting_data/train/def_front
OK class folder: /content/casting_dataset/casting_data/casting_data/train/ok_front
Defective images: 3758
OK images: 2875


## 7. Load libraries and create training/validation datasets

Images are resized to 224 × 224. The dataset is split into training and validation sets.

In [8]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Build a temporary directory containing only the two selected class folders.
# This avoids accidentally mixing other folders from the ZIP into the labels.
PREPARED_DIR = Path("/content/casting_binary_dataset")
if PREPARED_DIR.exists():
    shutil.rmtree(PREPARED_DIR)
PREPARED_DIR.mkdir(parents=True, exist_ok=True)

for source_dir, class_name in [(DEFECT_DIR, "defective"), (OK_DIR, "ok")]:
    destination = PREPARED_DIR / class_name
    destination.mkdir(parents=True, exist_ok=True)
    for image_path in source_dir.iterdir():
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            shutil.copy2(image_path, destination / image_path.name)

train_ds = tf.keras.utils.image_dataset_from_directory(
    PREPARED_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    PREPARED_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

class_names = train_ds.class_names
print("Class order used by the model:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

Found 6633 files belonging to 2 classes.
Using 5307 files for training.
Found 6633 files belonging to 2 classes.
Using 1326 files for validation.
Class order used by the model: ['defective', 'ok']


## 8. Build the image-classification model

A compact CNN is used as a beginner-friendly baseline. It predicts whether an image is `defective` or `ok`.

In [9]:
model = keras.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Rescaling(1.0 / 255),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 101,569 (396.75 KB)

 Trainable params: 101,569 (396.75 KB)

 Non-trainable params: 0 (0.00 B)

## 9. Train the model

The model trains for up to 8 epochs. Early stopping keeps the best validation result and stops if validation loss does not improve.

In [10]:
OUTPUT_DIR = Path("/content/casting_model_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUTPUT_DIR / "casting_defect_classifier.keras"),
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks
)

Epoch 1/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 617s 4s/step - accuracy: 0.5670 - loss: 0.6874 - val_accuracy: 0.5581 - val_loss: 0.6884
Epoch 2/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 606s 4s/step - accuracy: 0.5687 - loss: 0.6820 - val_accuracy: 0.5581 - val_loss: 0.6831
Epoch 3/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 592s 4s/step - accuracy: 0.5687 - loss: 0.6819 - val_accuracy: 0.5581 - val_loss: 0.6819
Epoch 4/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 633s 4s/step - accuracy: 0.5666 - loss: 0.6802 - val_accuracy: 0.5581 - val_loss: 0.6779
Epoch 5/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 594s 4s/step - accuracy: 0.5704 - loss: 0.6743 - val_accuracy: 0.6026 - val_loss: 0.6758
Epoch 6/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 617s 4s/step - accuracy: 0.5760 - loss: 0.6581 - val_accuracy: 0.5581 - val_loss: 0.6263
Epoch 7/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 623s 4s/step - accuracy: 0.6586 - loss: 0.5844 - val_accuracy: 0.7594 - val_loss: 0.5470
Epoch 8/8
166/166 ━━━━━━━━━━━━━━━━━━━━ 630s 4s/step - accuracy: 0.7017 - loss: 0.5274 - val_accuracy: 0.

## 10. Evaluate the model

In [12]:
import json
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.4f}")

y_true = np.concatenate([labels.numpy().reshape(-1) for _, labels in val_ds]).astype(int)
y_prob = model.predict(val_ds, verbose=0).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)
print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    zero_division=0
))
print("Confusion matrix:")
print(confusion_matrix(y_true, y_pred))

metrics = {
    "dataset": "Real-Life Industrial Dataset of Casting Product",
    "class_names": class_names,
    "validation_loss": float(val_loss),
    "validation_accuracy": float(val_accuracy),
    "classification_report": report,
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
}

with open(OUTPUT_DIR / "evaluation_metrics.json", "w") as file:
    json.dump(metrics, file, indent=2)

Validation loss: 0.4894
Validation accuracy: 0.7858

Classification report:
              precision    recall  f1-score   support

   defective       0.59      0.37      0.45       740
          ok       0.46      0.68      0.55       586

    accuracy                           0.50      1326
   macro avg       0.52      0.52      0.50      1326
weighted avg       0.53      0.50      0.49      1326

Confusion matrix:
[[271 469]
 [189 397]]


## 11. Save the final model and training history

In [13]:
import pandas as pd

# Save the final model in Keras format.
model.save(OUTPUT_DIR / "casting_defect_classifier_final.keras")

history_df = pd.DataFrame(history.history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

# Save class names so the prediction label order is documented.
with open(OUTPUT_DIR / "class_names.json", "w") as file:
    json.dump(class_names, file, indent=2)

print("Saved output files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name, "-", round(path.stat().st_size / (1024 * 1024), 2), "MB")

Saved output files:
casting_defect_classifier.keras - 1.21 MB
casting_defect_classifier_final.keras - 1.21 MB
class_names.json - 0.0 MB
evaluation_metrics.json - 0.0 MB
training_history.csv - 0.0 MB


In [20]:
val_ds_fixed = tf.keras.utils.image_dataset_from_directory(
      PREPARED_DIR,
          validation_split=0.2,
              subset="validation",
                  seed=SEED,
                      image_size=IMG_SIZE,
                          batch_size=BATCH_SIZE,
                              label_mode="binary",
                                  shuffle=False
                                  )

                                  class_names = val_ds_fixed.class_names

                                  val_ds_fixed = val_ds_fixed.prefetch(tf.data.AUTOTUNE)

                                  val_loss, val_accuracy = model.evaluate(
                                      val_ds_fixed,
                                          verbose=0
                                          )

                                          y_true = np.concatenate([
                                              labels.numpy().reshape(-1)
                                                  for images, labels in val_ds_fixed
                                                  ]).astype(int)

                                                  y_prob = model.predict(
                                                      val_ds_fixed,
                                                          verbose=0
                                                          ).reshape(-1)

                                                          y_pred = (y_prob >= 0.5).astype(int)

                                                          from sklearn.metrics import classification_report, confusion_matrix

                                                          print("Validation loss:", round(val_loss, 4))
                                                          print("Validation accuracy:", round(val_accuracy, 4))

                                                          print("\nClassification Report:")
                                                          print(classification_report(
                                                              y_true,
                                                                  y_pred,
                                                                      target_names=class_names,
                                                                          zero_division=0
                                                                          ))

                                                                          print("Confusion Matrix:")
                                                                          print(confusion_matrix(y_true, y_pred))
)

Found 6633 files belonging to 2 classes.
Using 1326 files for validation.
Validation loss: 0.463
Validation accuracy: 0.9947

Classification Report:
              precision    recall  f1-score   support

   defective       0.00      0.00      0.00         0
          ok       1.00      0.99      1.00      1326

    accuracy                           0.99      1326
   macro avg       0.50      0.50      0.50      1326
weighted avg       1.00      0.99      1.00      1326

Confusion Matrix:
[[   0    0]
 [   7 1319]]


In [21]:
import numpy as np

print("Class names:", class_names)

print("\nActual validation labels:")
print(np.unique(y_true, return_counts=True))

print("\nPredicted labels:")
print(np.unique(y_pred, return_counts=True))

Class names: ['defective', 'ok']

Actual validation labels:
(array([1]), array([1326]))

Predicted labels:
(array([0, 1]), array([   7, 1319]))


In [24]:
import os

print("PREPARED_DIR:", PREPARED_DIR)
print("\nFolders inside PREPARED_DIR:")

for root, dirs, files in os.walk(PREPARED_DIR):
    level = root.replace(PREPARED_DIR, "").count(os.sep)

        if level <= 2:
                print(
                            "  " * level,
                                        os.path.basename(root),
                                                    "| Images/files:",
                                                                len(files)
                                                                        )

PREPARED_DIR: /content/casting_binary_dataset

Folders inside PREPARED_DIR:
 casting_binary_dataset | Images/files: 0
   defective | Images/files: 3758
   ok | Images/files: 2875


In [26]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split

defective_dir = os.path.join(PREPARED_DIR, "defective")
ok_dir = os.path.join(PREPARED_DIR, "ok")

defective_files = [
    os.path.join(defective_dir, f)
        for f in os.listdir(defective_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
            ]

            ok_files = [
                os.path.join(ok_dir, f)
                    for f in os.listdir(ok_dir)
                        if f.lower().endswith((".jpg", ".jpeg", ".png"))
                        ]

                        file_paths = defective_files + ok_files
                        labels = [0] * len(defective_files) + [1] * len(ok_files)

                        train_paths, val_paths, train_labels, val_labels = train_test_split(
                            file_paths,
                                labels,
                                    test_size=0.2,
                                        random_state=42,
                                            stratify=labels
                                            )

                                            print("Training images:", len(train_paths))
                                            print("Validation images:", len(val_paths))
                                            print("Defective validation images:", val_labels.count(0))
                                            print("OK validation images:", val_labels.count(1))

Training images: 5306
Validation images: 1327
Defective validation images: 752
OK validation images: 575


In [30]:
import numpy as np
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def load_image(path, label):
    image = tf.io.read_file(path)
        image = tf.io.decode_image(
                image,
                        channels=3,
                                expand_animations=False
                                    )
                                        image = tf.image.resize(image, IMG_SIZE)
                                            image = tf.cast(image, tf.float32)
                                                return image, label

                                                val_ds_fixed = tf.data.Dataset.from_tensor_slices(
                                                    (val_paths, val_labels)
                                                    )

                                                    val_ds_fixed = val_ds_fixed.map(
                                                        load_image,
                                                            num_parallel_calls=tf.data.AUTOTUNE
                                                            )

                                                            val_ds_fixed = val_ds_fixed.batch(BATCH_SIZE).prefetch(
                                                                tf.data.AUTOTUNE
                                                                )

                                                                y_true = np.array(val_labels, dtype=int)

                                                                y_prob = model.predict(
                                                                    val_ds_fixed,
                                                                        verbose=1
                                                                        ).reshape(-1)

                                                                        y_pred = (y_prob >= 0.5).astype(int)

                                                                        print("\nClassification Report:")
                                                                        print(
                                                                            classification_report(
                                                                                    y_true,
                                                                                            y_pred,
                                                                                                    labels=[0, 1],
                                                                                                            target_names=["defective", "ok"],
                                                                                                                    zero_division=0
                                                                                                                        )
                                                                                                                        )

                                                                                                                        print("Confusion Matrix:")
                                                                                                                        print(
                                                                                                                            confusion_matrix(
                                                                                                                                    y_true,
                                                                                                                                            y_pred,
                                                                                                                                                    labels=[0, 1]
                                                                                                                                                        )
                                                                                                                                                        )

42/42 ━━━━━━━━━━━━━━━━━━━━ 44s 998ms/step

Classification Report:
              precision    recall  f1-score   support

   defective       0.99      0.61      0.76       752
          ok       0.66      0.99      0.80       575

    accuracy                           0.78      1327
   macro avg       0.83      0.80      0.78      1327
weighted avg       0.85      0.78      0.77      1327

Confusion Matrix:
[[462 290]
 [  4 571]]


In [32]:
import json
import os
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

metrics = {
    "model": "Casting Defect Classification CNN",
        "dataset": "Real-Life Industrial Dataset of Casting Product",
            "validation_images": int(len(y_true)),
                "accuracy": float(accuracy_score(y_true, y_pred)),
                    "classification_report": classification_report(
                            y_true,
                                    y_pred,
                                            labels=[0, 1],
                                                    target_names=["defective", "ok"],
                                                            output_dict=True,
                                                                    zero_division=0
                                                                        ),
                                                                            "confusion_matrix": confusion_matrix(
                                                                                    y_true,
                                                                                            y_pred,
                                                                                                    labels=[0, 1]
                                                                                                        ).tolist()
                                                                                                        }

                                                                                                        output_dir = "/content/casting_defect_classification_package"
                                                                                                        os.makedirs(output_dir, exist_ok=True)

                                                                                                        with open(
                                                                                                            os.path.join(output_dir, "evaluation_metrics.json"),
                                                                                                                "w"
                                                                                                                ) as f:
                                                                                                                    json.dump(metrics, f, indent=4)

                                                                                                                    print("Corrected evaluation metrics saved successfully!")
                                                                                                                    print("Accuracy:", round(metrics["accuracy"] * 100, 2), "%")

Corrected evaluation metrics saved successfully!
Accuracy: 77.84 %


## 12. Test the model on one image (optional)

Upload an image when prompted. The model prints the predicted class and confidence. This is a demonstration, not a guarantee of industrial quality.

In [33]:
from google.colab import files
from PIL import Image
import numpy as np

test_upload = files.upload()

for filename in test_upload:
    if Path(filename).suffix.lower() not in IMAGE_EXTENSIONS:
        print("Skipping non-image file:", filename)
        continue

    image = Image.open(filename).convert("RGB").resize(IMG_SIZE)
    image_array = np.array(image, dtype=np.float32)[None, ...]

    probability_class_1 = float(model.predict(image_array, verbose=0)[0][0])
    predicted_index = 1 if probability_class_1 >= 0.5 else 0
    confidence = probability_class_1 if predicted_index == 1 else 1 - probability_class_1

    print("Image:", filename)
    print("Predicted class:", class_names[predicted_index])
    print("Confidence:", f"{confidence:.2%}")

KeyboardInterrupt: 

## 13. Package the model files and results for GitHub

This creates a ZIP that you can extract locally and upload into a new folder in your project repository. It does not push anything to GitHub automatically.

In [35]:
import shutil
from pathlib import Path

PACKAGE_DIR = Path("/content/casting_defect_classification_package")
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

# Copy model and result files into a clean package.
for filename in [
    "casting_defect_classifier.keras",
    "casting_defect_classifier_final.keras",
    "evaluation_metrics.json",
    "training_history.csv",
    "class_names.json"
]:
    source = OUTPUT_DIR / filename
    if source.exists():
        shutil.copy2(source, PACKAGE_DIR / filename)

readme_text = f\'''# Casting Defect Classification

- Dataset: Real-Life Industrial Dataset of Casting Product (Kaggle)
- Task: Binary image classification (`defective` vs `ok`)
- Image size: 224 x 224
- Validation accuracy: {val_accuracy:.4f}
- Class order: {class_names}

## Files
- `casting_defect_classifier_final.keras`: final trained Keras model
- `casting_defect_classifier.keras`: best checkpoint saved during training
- `evaluation_metrics.json`: validation metrics and classification report
- `training_history.csv`: training and validation history
- `class_names.json`: class order used by the model

Note: Validation results are from this notebook run. Performance may vary with a different split or dataset version. This is a project prototype, not a certified industrial inspection system.
\'''

(PACKAGE_DIR / "README.md").write_text(readme_text, encoding="utf-8")

PACKAGE_ZIP = shutil.make_archive(
    "/content/casting_defect_classification_package",
    "zip",
    root_dir=PACKAGE_DIR
)

print("Package created:", PACKAGE_ZIP)
files.download(PACKAGE_ZIP)

Package created: /content/casting_defect_classification_package.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>